In [1]:
import sys
import os

ROOT = os.path.abspath("..")  # adjust if needed
print(ROOT)
OPENPCDET_PATH = '/storage/project/r-gchou3-0/spanse30/OpenPCDet'
print(OPENPCDET_PATH)
# # Put OpenPCDet FIRST
sys.path.insert(0, OPENPCDET_PATH)

# # Import OpenPCDet pcdet
import pcdet
print("Using pcdet from:", pcdet.__file__)

/storage/scratch1/9/spanse30
/storage/project/r-gchou3-0/spanse30/OpenPCDet
Using pcdet from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/__init__.py


In [2]:
import pickle as pkl

import os
from pathlib import Path
import copy
import numpy as np
import torch
    

In [3]:
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.models import build_network, load_data_to_gpu
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils
import random

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

import importlib
import helpers_ptt
import removal
importlib.reload(helpers_ptt)
importlib.reload(removal)
import torch.nn.functional as F

from collections import defaultdict

In [4]:
from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()

CFG_FILE_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml' 
CKPT_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth'  # <- put your ckpt here

logger = common_utils.create_logger()
logger.info(f'Loaded cfg from {CFG_FILE_MF}')

cfg_from_yaml_file(
    CFG_FILE_MF,
    cfg
)

cfg.TAG = 'centerpoint_multiframe'
cfg.EXP_GROUP_PATH = 'waymo_multiframe'

dataset_mf, test_loader_mf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger,
    training=False
)

len_test = len(dataset_mf)
logger.info(f'Test set length: {len_test}')
cp_mf_model = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset_mf
)

logger.info(f'Loading checkpoint from: {CKPT_MF}')
cp_mf_model.load_params_from_file(filename=CKPT_MF, logger=logger, to_cpu=False)
cp_mf_model.cuda()
cp_mf_model.eval()

data_iter_mf = iter(test_loader_mf)
batch_dict_mf = next(data_iter_mf)

load_data_to_gpu(batch_dict_mf)

2026-04-03 18:47:30,347   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml
2026-04-03 18:47:30,372   INFO  Loading Waymo dataset
2026-04-03 18:47:33,635   INFO  Total skipped info 0
2026-04-03 18:47:33,636   INFO  Total samples for Waymo dataset: 39987
2026-04-03 18:47:33,637   INFO  Test set length: 39987
2026-04-03 18:47:33,900   INFO  Loading checkpoint from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth
2026-04-03 18:47:33,904   INFO  ==> Loading parameters from checkpoint /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth to GPU
2026-04-03 18:47:34,163   INFO  ==> Checkpoint trained from version: pcdet+0.6.0+233f849
2026-04-03 18:47:34,174   INFO  ==> Done (loaded 288/288)


In [5]:
from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()


CFG_FILE_SF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml' 
CKPT_SF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth'  # <- put your ckpt here

cfg_from_yaml_file(CFG_FILE_SF, cfg)
cfg.TAG = Path(CFG_FILE_SF).stem
cfg.EXP_GROUP_PATH = 'centerpoint_waymo_demo'

import os
# cfg.DATA_CONFIG.DATA_PATH = os.path.join(OPENPCDET_PATH, 'data', 'waymo')

logger_sf = common_utils.create_logger()
logger_sf.info(f'Loaded cfg from {CFG_FILE_SF}')


dataset_sf, test_loader_sf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger_sf,
    training=False
)

len_test_sf = len(dataset_sf)
logger.info(f'Test set length: {len_test_sf}')
model_sf = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset_sf
)

logger.info(f'Loading checkpoint from: {CKPT_SF}')
model_sf.load_params_from_file(filename=CKPT_SF, logger=logger, to_cpu=False)
model_sf.cuda()
model_sf.eval()

data_iter_sf = iter(test_loader_sf)
batch_dict_sf = next(data_iter_sf)

load_data_to_gpu(batch_dict_sf)



2026-04-03 18:47:35,718   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml
2026-04-03 18:47:35,718   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml
2026-04-03 18:47:35,737   INFO  Loading Waymo dataset
2026-04-03 18:47:35,737   INFO  Loading Waymo dataset
2026-04-03 18:47:37,743   INFO  Total skipped info 0
2026-04-03 18:47:37,743   INFO  Total skipped info 0
2026-04-03 18:47:37,751   INFO  Total samples for Waymo dataset: 38597
2026-04-03 18:47:37,751   INFO  Total samples for Waymo dataset: 38597
2026-04-03 18:47:37,753   INFO  Test set length: 38597
2026-04-03 18:47:37,753   INFO  Test set length: 38597
2026-04-03 18:47:37,807   INFO  Loading checkpoint from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth
2026-04-03 18:47:37,807   INFO  Loading checkpoint from: /storage

In [6]:
# Extract frame IDs from the dataset metadata
# For Waymo, frame_id is usually formatted as 'segment_name_frame_idx'
frame_ids = [info['frame_id'] for info in dataset_sf.infos]

# Split the frame_id to get only the segment name (everything before the last underscore)
# and use set() to find unique values
unique_segments_sf = sorted(list(set(fid.rsplit('_', 1)[0] for fid in frame_ids)))

In [9]:
dataset_sf[0].keys()

dict_keys(['sample_idx', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'voxels', 'voxel_coords', 'voxel_num_points', 'metadata'])

In [11]:
dataset_sf[0]['sample_idx']

0

In [13]:
infos = pkl.load(open("/storage/project/r-gchou3-0/shared/waymo_processed_data_v0_5_0_infos_val.pkl", "rb"))

print(infos[0].keys())

dict_keys(['point_cloud', 'frame_id', 'metadata', 'image', 'pose', 'annos', 'num_points_of_each_lidar'])


In [20]:
infos[0]['num_points_of_each_lidar']

[112032, 3264, 3662, 3787, 3759]

In [7]:
importlib.reload(removal)


<module 'removal' from '/storage/project/r-gchou3-0/spanse30/OpenPCDet/removal.py'>

In [15]:
importlib.reload(helpers_ptt)


<module 'helpers_ptt' from '/storage/project/r-gchou3-0/spanse30/OpenPCDet/helpers_ptt.py'>

In [16]:
import numpy as np


def get_spoof_subsets(dataset, block_size=32):
    """
    Compute (global_start, global_end) tuples for all spoofed subsets
    across all segments in the dataset.

    Pattern per segment (0-indexed within segment):
        clean:   [0, 32)
        spoof:   [32, 64)
        clean:   [64, 96)
        spoof:   [96, 128)
        clean:   [128, 160)
        spoof:   [160, 192)
        clean:   [192, ...)   (leftover if any)

    So spoofed blocks start at segment-relative indices: 32, 96, 160, ...
    i.e., every other block starting from block index 1.

    Parameters
    ----------
    dataset : dataset object with .infos list
        Each entry in dataset.infos should have a 'sequence_name' or 
        'sample_idx' or similar field to identify segments.
    block_size : int
        Size of each clean/spoof block (default 32).

    Returns
    -------
    spoof_ranges : list of (global_start, global_end) tuples (inclusive end)
    segment_info : list of dicts with per-segment breakdown
    """
    # --- Step 1: identify segments and their global frame ranges ---
    infos = dataset.infos
    n_total = len(infos)

    # Extract segment name from frame_id by stripping the trailing _XXX frame number
    # e.g. 'segment-10203656353524179475_7625_000_7645_000_with_camera_labels_000'
    #   -> 'segment-10203656353524179475_7625_000_7645_000_with_camera_labels'
    def _seg_name(info):
        fid = info['frame_id']
        return fid.rsplit('_', 1)[0]

    # Build ordered list of (segment_name, global_start, global_end)
    segments = []
    current_seg = None
    seg_start = 0

    for i in range(n_total):
        seg_name = _seg_name(infos[i])

        if seg_name != current_seg:
            if current_seg is not None:
                segments.append({
                    "name": current_seg,
                    "global_start": seg_start,
                    "global_end": i - 1,
                    "n_frames": i - seg_start,
                })
            current_seg = seg_name
            seg_start = i

    # Don't forget last segment
    segments.append({
        "name": current_seg,
        "global_start": seg_start,
        "global_end": n_total - 1,
        "n_frames": n_total - seg_start,
    })

    # --- Step 2: compute spoof subsets per segment ---
    spoof_ranges = []
    segment_info = []

    for seg in segments:
        g_start = seg["global_start"]
        n_frames = seg["n_frames"]

        # Spoofed blocks are at segment-relative positions:
        # [1*block, 2*block), [3*block, 4*block), [5*block, 6*block), ...
        # i.e., odd-numbered blocks (0-indexed)
        seg_spoof_ranges = []
        block_idx = 0
        pos = 0
        while pos < n_frames:
            block_end = min(pos + block_size, n_frames)
            if block_idx % 2 == 1:  # odd block = spoofed
                global_s = g_start + pos
                global_e = g_start + block_end - 1  # inclusive
                spoof_ranges.append((global_s, global_e))
                seg_spoof_ranges.append((global_s, global_e))
            pos = block_end
            block_idx += 1

        seg["spoof_ranges"] = seg_spoof_ranges
        segment_info.append(seg)

    return spoof_ranges, segment_info


def print_spoof_subsets(spoof_ranges, segment_info):
    """Pretty-print the spoof subset layout."""
    print(f"Total segments: {len(segment_info)}")
    print(f"Total spoof subsets: {len(spoof_ranges)}")
    print()

    for seg in segment_info:
        print(f"Segment: {seg['name']}")
        print(f"  Global frames: [{seg['global_start']}, {seg['global_end']}] "
              f"({seg['n_frames']} frames)")
        for s, e in seg['spoof_ranges']:
            n = e - s + 1
            rel_s = s - seg['global_start']
            rel_e = e - seg['global_start']
            full = "full" if n == 32 else f"TRUNCATED ({n} frames)"
            print(f"  Spoof block: global [{s}, {e}], "
                  f"segment-relative [{rel_s}, {rel_e}], {full}")
        print()

    print("All spoof ranges (global indices, inclusive):")
    for i, (s, e) in enumerate(spoof_ranges):
        print(f"  [{i}] frames {s}–{e} ({e - s + 1} frames)")


# ---- Usage ----
spoof_ranges, segment_info = get_spoof_subsets(dataset_sf)
spoof_ranges[2]

(160, 191)

In [17]:
import numpy as np

def track_sequence(dataset, start=32, end=63, desired_range=30.0,
                   target_class=1, max_step=6.0, negative = False):
    """
    Track a single GT object across frames.

    Parameters
    ----------
    dataset : indexable sequence where dataset[i] has keys
              'gt_boxes' (N,7+) and optionally 'gt_labels' (N,)
    start, end : int — frame range (inclusive)
    desired_range : float — preferred initial target range
    target_class : int — object class to track (1=vehicle)
    max_step : float — matching gate in meters

    Returns
    -------
    tracked_boxes : list of (box or None) per frame
    centers : (T, 3) array with NaN for lost frames
    diagnostics : dict
    """
    T = end - start + 1
    tracked_boxes = [None] * T
    tracked_idxs = [None] * T
    centers = np.full((T, 3), np.nan, dtype=float)
    match_dists = np.full(T, np.nan)
    ranges = np.full(T, np.nan)

    # --- Init ---
    init_data = dataset[start]
    init_boxes = init_data['gt_boxes']

    init_idx, cur_box = removal.select_initial_target(
        init_boxes,
        desired_range=desired_range, target_class=target_class, negative=negative
    )
    if cur_box is None:
        return tracked_boxes, centers, {"ok": False, "reason": "no_init_target"}

    tracked_boxes[0] = cur_box
    tracked_idxs[0] = init_idx
    centers[0] = cur_box[:3]
    ranges[0] = removal._box_range(cur_box)

    prev_box = cur_box
    prev_prev_box = None

    # --- Track ---
    for t in range(1, T):
        frame_data = dataset[start + t]
        curr_boxes = frame_data['gt_boxes']

        new_box, new_idx, d = removal.match_with_velocity(
            prev_box, prev_prev_box, curr_boxes,
            target_class=target_class,
            max_step=max_step
        )

        if new_box is None:
            # Lost — stop tracking
            break

        tracked_boxes[t] = new_box
        tracked_idxs[t] = new_idx
        centers[t] = new_box[:3]
        match_dists[t] = d
        ranges[t] = removal._box_range(new_box)

        prev_prev_box = prev_box
        prev_box = new_box

    # --- Diagnostics ---
    valid = ~np.isnan(centers[:, 0])
    num_tracked = int(np.sum(valid))

    deltas = np.linalg.norm(np.diff(centers, axis=0), axis=1)
    speeds_mps = deltas * 10.0  # approximate if 10 Hz

    diagnostics = {
        "ok": True,
        "num_tracked": num_tracked,
        "total_frames": T,
        "init_range_m": float(ranges[0]),
        "range_trend": ranges[valid],
        "match_dists": match_dists,
        "step_deltas_m": deltas,
        "approx_speeds_mps": speeds_mps,
        "max_step_m": float(np.nanmax(deltas)) if num_tracked > 1 else 0.0,
        "max_speed_mps": float(np.nanmax(speeds_mps)) if num_tracked > 1 else 0.0,
        "mean_match_dist_m": float(np.nanmean(match_dists)) if num_tracked > 1 else 0.0,
    }
    
    return tracked_boxes, centers, diagnostics

# def track_sequence(dataset, start=32, end=63,
#                    desired_range=60.0,
#                    target_class=1,
#                    max_step=6.0,
#                    negative=False):

#     T = end - start + 1
#     tracked_boxes = [None] * T
#     centers = np.full((T, 3), np.nan)
#     match_dists = np.full(T, np.nan)
#     ranges = np.full(T, np.nan)

#     init_boxes = dataset[start]['gt_boxes']

#     init_idx, cur_box = removal.select_initial_target(
#         init_boxes,
#         desired_range=desired_range,
#         target_class=target_class,
#         negative=negative
#     )

#     if cur_box is None:
#         return tracked_boxes, centers, {"ok": False}

#     tracked_boxes[0] = cur_box
#     centers[0] = cur_box[:3]
#     ranges[0] = np.linalg.norm(cur_box[:3])

#     prev_box = cur_box
#     prev_prev_box = None

#     for t in range(1, T):
#         curr_boxes = dataset[start + t]['gt_boxes']

#         new_box, _, d = removal.match_with_velocity(
#             prev_box, prev_prev_box, curr_boxes,
#             max_step=max_step
#         )

#         if new_box is None:
#             break

#         tracked_boxes[t] = new_box
#         centers[t] = new_box[:3]
#         match_dists[t] = d
#         ranges[t] = np.linalg.norm(new_box[:3])

#         prev_prev_box = prev_box
#         prev_box = new_box

#     valid = ~np.isnan(centers[:, 0])
#     deltas = np.linalg.norm(np.diff(centers[valid], axis=0), axis=1) if np.sum(valid) > 1 else np.array([])

#     diagnostics = {
#         "ok": True,
#         "ranges": ranges[valid],
#         "init_range": ranges[0],
#         "num_tracked": int(np.sum(valid)),
#         "max_range": float(np.nanmax(ranges)),
#         "min_range": float(np.nanmin(ranges)),
#         "mean_match_dist": float(np.nanmean(match_dists)),
#         "max_step": float(np.nanmax(deltas)) if len(deltas) else 0.0
#     }

#     return tracked_boxes, centers, diagnostics

import numpy as np
import matplotlib.pyplot as plt


def debug_track_sequence(dataset, start=32, end=63, desired_range=30.0,
                         target_class=1, max_step=6.0, negative=False):
    """
    Wraps track_sequence with diagnostic prints and plots.
    """

    # ---- 1. Pre-tracking: inspect what's available at init frame ----
    init_data = dataset[start]
    init_boxes = init_data['gt_boxes']

#     print(f"=== INIT FRAME {start} ===")
#     print(f"  Total GT boxes: {len(init_boxes)}")

#     if len(init_boxes) > 0:
#         labels = init_boxes[:, -1].astype(int)
#         unique, counts = np.unique(labels, return_counts=True)
#         print(f"  Label distribution: {dict(zip(unique, counts))}")
#         class_mask = (labels == target_class)
#         class_boxes = init_boxes[class_mask]
#         print(f"  Boxes with target_class={target_class}: {len(class_boxes)}")
#     else:
#         print("  No GT boxes in init frame")
#         class_boxes = init_boxes

#     if len(class_boxes) > 0:
#         dists = np.linalg.norm(class_boxes[:, :3], axis=1)
#         in_front = class_boxes[:, 0] > 0
#         print(f"  Class {target_class} ranges: {np.sort(dists).round(1)}")
#         print(f"  Class {target_class} in front (x>0): {int(in_front.sum())}/{len(class_boxes)}")
#         print(f"  Desired range: {desired_range} ± 5 m")

    # ---- 2. Run tracking ----
    tracked_boxes, centers, diag = debug_track_sequence_inner(
        dataset, start, end, desired_range, target_class, max_step, negative
    )

    return tracked_boxes, centers, diag


def debug_track_sequence_inner(dataset, start, end, desired_range,
                                target_class, max_step, negative):
    """Run track_sequence and print per-frame diagnostics."""

    tracked_boxes, centers, diag = track_sequence(
        dataset, start=start, end=end, desired_range=desired_range,
        target_class=target_class, max_step=max_step, negative=negative
    )

#     if not diag["ok"]:
#         print(f"\n  TRACKING FAILED: {diag['reason']}")
#         return tracked_boxes, centers, diag

#     T = end - start + 1

#     # ---- 3. Per-frame summary ----
#     print(f"\n=== PER-FRAME TRACKING LOG ===")
#     print(f"  {'frame':>6s} | {'range':>7s} | {'match_d':>7s} | {'step':>6s} | {'vol':>7s} | {'center (x,y,z)'}")
#     print(f"  {'-'*6} | {'-'*7} | {'-'*7} | {'-'*6} | {'-'*7} | {'-'*30}")

#     for t in range(T):
#         frame_idx = start + t
#         box = tracked_boxes[t]

#         if box is None:
#             print(f"  {frame_idx:6d} | {'LOST':>7s} |")
#             continue

#         r = helpers_ptt._box_range(box)
#         vol = helpers_ptt._box_volume(box)
#         md = diag["match_dists"][t]
#         step = diag["step_deltas_m"][t - 1] if t > 0 else float('nan')
#         ctr = box[:3]

#         md_str = f"{md:.3f}" if not np.isnan(md) else "init"
#         step_str = f"{step:.3f}" if not np.isnan(step) else "-"

#         print(f"  {frame_idx:6d} | {r:7.2f} | {md_str:>7s} | {step_str:>6s} | {vol:7.2f} | ({ctr[0]:.1f}, {ctr[1]:.1f}, {ctr[2]:.1f})")

#     # ---- 4. Summary stats ----
#     print(f"\n=== TRACKING SUMMARY ===")
#     print(f"  Frames tracked: {diag['num_tracked']}/{T}")
#     print(f"  Init range:     {diag['init_range_m']:.1f} m")
#     print(f"  Final range:    {helpers_ptt._box_range(tracked_boxes[diag['num_tracked']-1]):.1f} m"
#           if tracked_boxes[diag['num_tracked']-1] is not None else "  Final range: LOST")
#     print(f"  Mean match dist: {diag['mean_match_dist_m']:.3f} m")
#     print(f"  Max step:        {diag['max_step_m']:.3f} m")
#     print(f"  Max speed:       {diag['max_speed_mps']:.1f} m/s ({diag['max_speed_mps']*3.6:.1f} km/h)")

#     # ---- 5. Sanity checks ----
#     print(f"\n=== SANITY CHECKS ===")
#     ranges = diag["range_trend"]

#     # Check: did range change monotonically? (approaching or receding)
#     if len(ranges) > 2:
#         diffs = np.diff(ranges)
#         mostly_approaching = np.sum(diffs < 0) > len(diffs) * 0.7
#         mostly_receding = np.sum(diffs > 0) > len(diffs) * 0.7
#         if mostly_approaching:
#             print(f"  Range trend: APPROACHING (range decreasing)")
#         elif mostly_receding:
#             print(f"  Range trend: RECEDING (range increasing)")
#         else:
#             print(f"  Range trend: MIXED — check if tracker jumped to another object")

#     # Check: any suspiciously large steps?
#     large_steps = np.where(diag["step_deltas_m"] > max_step * 0.8)[0]
#     if len(large_steps) > 0:
#         print(f"  WARNING: {len(large_steps)} frames had steps near max_step gate ({max_step}m)")
#         print(f"           Frame indices: {[start + 1 + i for i in large_steps]}")
#     else:
#         print(f"  Step sizes: all well within gate ({max_step}m)")

#     # Check: volume consistency
#     vols = np.array([helpers_ptt._box_volume(b) for b in tracked_boxes if b is not None])
#     if len(vols) > 1:
#         vol_cv = np.std(vols) / np.mean(vols)
#         if vol_cv > 0.3:
#             print(f"  WARNING: box volume varies a lot (CV={vol_cv:.2f}) — possible ID switch")
#         else:
#             print(f"  Box volume: stable (CV={vol_cv:.2f})")

#     # ---- 6. Plots ----
#     fig, axes = plt.subplots(1, 3, figsize=(15, 4))

#     # BEV trajectory
#     valid = ~np.isnan(centers[:, 0])
#     axes[0].plot(centers[valid, 0], centers[valid, 1], '-o', markersize=3)
#     axes[0].set_aspect('equal', 'box')
#     axes[0].set_title("Tracked trajectory (BEV)")
#     axes[0].set_xlabel("x (m)")
#     axes[0].set_ylabel("y (m)")

#     # Range over time
#     frames = np.arange(start, end + 1)
#     axes[1].plot(frames[valid], ranges, '-o', markersize=3)
#     axes[1].set_title("Range from ego over time")
#     axes[1].set_xlabel("Frame")
#     axes[1].set_ylabel("Range (m)")

#     # Match distance over time
#     md_valid = ~np.isnan(diag["match_dists"])
#     axes[2].plot(frames[md_valid], diag["match_dists"][md_valid], '-o', markersize=3)
#     axes[2].axhline(y=max_step, color='r', linestyle='--', alpha=0.5, label=f'gate={max_step}m')
#     axes[2].set_title("Match distance per frame")
#     axes[2].set_xlabel("Frame")
#     axes[2].set_ylabel("Match dist (m)")
#     axes[2].legend()

#     plt.tight_layout()
#     plt.show()

    return tracked_boxes, centers, diag


# for i in range(0, 585):
#     tracked_boxes, centers, diag = debug_track_sequence(
#         dataset_mf, start=spoof_ranges[i][0], end=spoof_ranges[i][1], desired_range=60.0, target_class=1, negative = True
#     )
#     mask = ~np.isnan(centers).any(axis=1)

#     # Get the indices
#     indices = np.where(mask)[0]

#     if indices.size > 0:
#         first_idx = indices[0]
#         last_idx = indices[-1]
#     else:
#         first_idx = last_idx = None # Handle the "all NaN" case
    
    # if(np.isnan(centers[0][0]) or (np.abs(centers[first_idx][0]-centers[last_idx][0]) < 10 and np.abs(centers[first_idx][0]) < 10)):
    #     print("fallback")
    #     tracked_boxes, centers, diag = debug_track_sequence(
    #         dataset_mf, start=spoof_ranges[i][0], end=spoof_ranges[i][1], desired_range=60.0, target_class=1, negative = True
    #     )
        
#         mask = ~np.isnan(centers).any(axis=1)

#         # Get the indices
#         indices = np.where(mask)[0]

#         if indices.size > 0:
#             first_idx = indices[0]
#             last_idx = indices[-1]
#         else:
#             first_idx = last_idx = None # Handle the "all NaN" case
            
            
        # if(np.isnan(centers[0][0])):
        #     ranges = [20, 30, 40, 50]
        #     for j in ranges:
        #         tracked_boxes, centers, diag = debug_track_sequence(
        #             dataset_mf, start=spoof_ranges[i][0], end=spoof_ranges[i][1], desired_range=j, target_class=1, negative = True
        #         )
        #         if(not np.isnan(centers[0][0])):
        #             print(f"range {j} works")
        #             break
        #     print("can't find any valid boxes...need to skip this segment")
        #     print(centers)
        #     break
    x_center = [center[0] for center in centers]
    
    mask = ~np.isnan(centers).any(axis=1)

    # 2. Count them
    num_valid_rows = mask.sum()
    print(spoof_ranges[i][0], spoof_ranges[i][1], max(x_center), min(x_center), num_valid_rows)

In [20]:
# mask = ~np.isnan(centers).any(axis=1)

# # Get the indices
# indices = np.where(mask)[0]

# if indices.size > 0:
#     first_idx = indices[0]
#     last_idx = indices[-1]
# else:
#     first_idx = last_idx = None # Handle the "all NaN" case

In [21]:
elevations = np.load('waymo_top_lidar_inclinations.npy')
stats = np.load("range_conditioned_stats.npy", allow_pickle=True).item()

In [22]:
def isolate_frame_points(pts, lag):

    # 2. Force a copy to ensure memory is contiguous
    pts_clean = np.array(pts, copy=True)
    # print(pts_clean.shape)
    # 3. Create a strict mask on the last column (index 5)
    # Using a gap-based threshold (0.05) since lags are 0, 0.1, 0.2, 0.3
    lags = np.round(pts_clean[:, -1], decimals=1)
    # print(len(lags))
    target_lag = 0.1*lag
    # mask = np.abs(pts_clean[:, 5]) < 0.05
    mask = (lags == target_lag)
    # print(mask)
    # 4. Apply
    current_frame = pts_clean[mask]
    # print(f"Verified Max Lag: {current_frame[:, 5].max()}")
    return current_frame


In [23]:
def process_4frames(dataset_mf,
                    dataset_sf, 
                    frame_idx, 
                    target_frames, 
                    frame_seq, 
                    removal_cache,
                    mf_to_sf_global):
    
    target_frames_copy = {k: v.copy() if v is not None else None for k,v in target_frames.items()}    
    #frame idx will be an index into centerpoint dataset
    # print(f"4frame point count before spoofing: {dataset_mf[frame_idx]['points'].shape}")
    points_frame = None
    #put global spoof in ego coordinates of lag = 0 frame
    none_spoofed = 1
    infos = dataset_mf.infos
    info_cur = infos[frame_idx]
    seq_name = info_cur['point_cloud']['lidar_sequence']
    pose_cur = info_cur['pose'].reshape(4, 4) #ego pose of current frame
    # print(seq_name)

    #acquire list of frames in set of 4 (centerpoint multiframe) that need to be inspected for modification
    indices = []
    for i in range(4):
        target_idx = frame_idx - i

        if target_idx < 0:
            target_idx = 0

        if infos[target_idx]['point_cloud']['lidar_sequence'] != seq_name:
            target_idx = indices[-1] if indices else frame_idx

        indices.append(target_idx)
            
    # print(f"cp 4 frames indices processed: {indices}")
    has_spoof= any(target_frames_copy[i] is not None for i in indices)
    #if none of the frames are marked for spoofing, just return the original points for the current frame (no raycasting needed)
    if not has_spoof:
        # print("not spoofing")
        pts = dataset_sf[mf_to_sf_global[frame_idx]]['points']
        points_frame = np.hstack([pts, np.zeros((pts.shape[0], 1))])
        return dataset_mf[frame_idx]['points'], none_spoofed, points_frame

    #iterate through 4 frames 
    merged_points_list = []
    for i, idx in enumerate(indices):
     
        
        #get points from single frame, add lag 0
        pts = dataset_sf[mf_to_sf_global[idx]]['points'].copy()
        points= np.hstack([pts, np.zeros((pts.shape[0], 1))])
        
        #update lag 0 if needed
        points[:, -1] = (frame_idx - idx)*0.1
        pose_lag_i_frame = infos[idx]['pose'].reshape(4, 4)

        

        if(target_frames_copy[idx] is not None):            
            ##???????????
            
            #use these for building the dataset
            # if idx == frame_idx, then this means this is lag 0, and it is already in lag 0 coordinates
            frame_points_spoof = removal_cache[idx].copy()
            frame_points_spoof[:, -1] = (frame_idx - idx) * 0.1  # set correct lag
            if(idx == frame_idx):
                points_frame = frame_points_spoof
                
                                                           
            # convert these back to lag 0 coordintaes 
            frame_points_spoof_hom = np.hstack((frame_points_spoof[:, :3], np.ones(frame_points_spoof.shape[0]).reshape(-1, 1)))
            frame_points_spoof_lag0_coords = frame_points_spoof_hom @ pose_lag_i_frame.T @ np.linalg.inv(pose_cur).T
                                                
            
            # print(f"spoofing frame {idx}, points diff = {frame_points_spoof_rc_lag0_coords.shape[0] - points.shape[0]}")
            points = np.hstack((frame_points_spoof_lag0_coords[:, :-1], frame_points_spoof[:, 3:]))
        else:
            # not spoofing, must convert back to lag 0 coordinates
            if(idx == frame_idx):
                points_frame = points
                
            points_last3cols = points[:, 3:]
            frame_points_hom = np.hstack((points[:, :3], np.ones(points.shape[0]).reshape(-1, 1)))
            frame_points_lag0_coords = frame_points_hom @ pose_lag_i_frame.T @ np.linalg.inv(pose_cur).T
            points = np.hstack((frame_points_lag0_coords[:, :-1], points_last3cols))
         
       
        # print(points.shape)
        merged_points_list.append(points)
    merged_points_array = np.vstack(merged_points_list)
    merged_points_array = merged_points_array.astype(np.float32)
    # print(f"4frame point count after spoofing: {merged_points_array.shape}")    

    
    none_spoofed = 0
    # unique_lags = np.unique(np.round(merged_points_array[:, -1], 1))
    # for lag in unique_lags:
    #     pts = merged_points_array[np.isclose(merged_points_array[:, -1], lag)]
    #     print(lag, pts.shape[0])
    # if frame_seq == 31:
    #     lags_raw = merged_points_array[:, -1]
    #     print("lag raw min/max:", lags_raw.min(), lags_raw.max())
    #     u, c = np.unique(np.round(lags_raw, 3), return_counts=True)
    #     print("unique lags (rounded 1e-3):")
    #     for uu, cc in zip(u[:30], c[:30]):
    #         print(uu, cc)
    #     print("... total unique:", len(u))
    #     # helpers_ptt.plot_4frame_sequence(merged_points_array)
    #     # input("Press Enter to continue to the next frame...")
    # print(n_spoof_r)
    # print(n_spoof_k)
    return merged_points_array, none_spoofed, points_frame

In [24]:
def generate_timing_classification(target_frames, window_size):
    classification = []

    keys = sorted(target_frames.keys())
    n_prev = window_size - 1

    for k in keys:
        cur = target_frames.get(k)

        prev_vals = [target_frames.get(k - i) for i in range(1, n_prev + 1)]

        prev_all_none = all(v is None for v in prev_vals)
        prev_any_not_none = any(v is not None for v in prev_vals)

        if cur is None and prev_all_none:
            classification.append(0)

        elif cur is not None and prev_all_none:
            classification.append(1)

        elif cur is not None and prev_any_not_none:
            classification.append(2)

        elif cur is None and prev_any_not_none:
            classification.append(3)

    return classification
# classification = generate_timing_classification(target_frames_seg['segment-10203656353524179475_7625_000_7645_000_with_camera_labels'])
# for i, c in enumerate(classification):
#     print(i, c)

In [25]:
# traces[0]['box']

In [26]:
def extract_one_runs(d):
    runs = []
    current_run = []

    for idx in sorted(d.keys()):
        if d[idx] == 1:
            current_run.append(idx)
        else:
            if current_run:
                runs.append(current_run)
                current_run = []

    # handle tail case
    if current_run:
        runs.append(current_run)

    return runs

In [27]:
from collections import defaultdict


mf_segments = set(info['point_cloud']['lidar_sequence']
                  for info in dataset_mf.infos)
common_segments = mf_segments.intersection(unique_segments_sf)

# Mapping: segment -> list of global indices (ordered)
segment_to_global = defaultdict(list)

# Mapping: global_idx -> (segment, local_idx)
global_to_segment = {}

for global_idx, info in enumerate(dataset_mf.infos):
    segment = info['point_cloud']['lidar_sequence']
    if segment in common_segments:
        local_idx = len(segment_to_global[segment])  # current position in segment
        segment_to_global[segment].append(global_idx)

        global_to_segment[global_idx] = (segment, local_idx)


segment_counts = defaultdict(int)

for info in dataset_mf.infos:
    segment = info['point_cloud']['lidar_sequence']
    if segment in common_segments: segment_counts[segment] += 1

segment_last_local_idx = {seg: count - 1 for seg, count in segment_counts.items()}


target_frames_global = {}
target_frames_seg = {}

for seg, last_ind in segment_last_local_idx.items():
    toggle = False
    targets_global = {}
    targets_seg = {}
    for i in range(last_ind+1):
        if i==0:
            targets_seg[i] = None
            targets_global[segment_to_global[seg][i]] = None
        else:
            rem = i%32
            # print(rem)
            if(not rem):
                toggle = not toggle
            if toggle:
                targets_seg[i] = 1
                targets_global[segment_to_global[seg][i]] = 1
            else:
                targets_seg[i] = None
                targets_global[segment_to_global[seg][i]] = None
    target_frames_global[seg] = targets_global
    target_frames_seg[seg] = targets_seg


sf_segment_to_global = defaultdict(list)

for global_idx, info in enumerate(dataset_sf.infos):
    seg = info['frame_id'].rsplit('_', 1)[0]
    if seg in common_segments:
        sf_segment_to_global[seg].append(global_idx)
mf_to_sf_global = {}

for mf_global, (seg, local_idx) in global_to_segment.items():
    if seg in sf_segment_to_global and local_idx < len(sf_segment_to_global[seg]):
        sf_global = sf_segment_to_global[seg][local_idx]
        mf_to_sf_global[mf_global] = sf_global

In [28]:
# target_frames_seg

In [29]:
dataset_sf[0]['gt_boxes'].shape

(10, 8)

In [30]:
dataset_mf[0]['gt_boxes'].shape

(10, 10)

In [ ]:
import time
start = time.time()

segment_dataset = []
save_dir_dataset = 'datasets/removal'
# os.makedirs(save_dir_dataset, exist_ok=True)
save_dir_annos = 'annos/removal'
# os.makedirs(save_dir_annos, exist_ok=True)

# completed = set(
#     f[:-4] for f in os.listdir(save_dir_dataset)
# )


rng = np.random.default_rng()
target_frames = {}
current_segment = None
target_frames_anno = {}
# prev_seq_name = None

prr = {10:0.97,
       20:1,
       30:0.99,
       40:0.98,
       50:0.9,
       60:0.9
}

try:
    for (frame,batch)  in enumerate(test_loader_mf):
        # if(frame <=230): continue
        segment = batch['frame_id'][0].rsplit("_", 1)[0]
        if segment not in common_segments:
            continue

#         if segment in completed:
#             continue

        if current_segment is None:
            print(segment)
            current_segment = segment
            target_frames = target_frames_global[current_segment].copy()
            target_frames_anno = target_frames_seg[current_segment].copy()


            #get spoofs for current segment
            # print(target_frames)
            runs_global = extract_one_runs(target_frames)            
            for run in runs_global:
                subset_g = {k: target_frames[k] for k in run}
                # subset_g_anno = {k: target_frames_anno[k] for k in run}
                window_min = min(subset_g)
                window_max = max(subset_g)

                tracked_boxes, centers, diag = track_sequence(
                    dataset_mf, start=window_min, end=window_max, desired_range=60.0, target_class=1, negative = True
                )
                ##debug####
                x_center = [center[0] for center in centers]

                mask = ~np.isnan(centers).any(axis=1)

                # 2. Count them
                num_valid_rows = mask.sum()
                print(window_min, window_max, max(x_center), min(x_center), num_valid_rows)

                ##end debug ##
                for key, center, box in zip(subset_g.keys(), centers, tracked_boxes):
                    subset_g[key] = box


                target_frames.update(subset_g)
            removal_cache = {}
            for run in runs_global:
                state = None
                for global_idx in sorted(run):
                    box = target_frames[global_idx]
                    if box is None:
                        target_frames_anno[global_to_segment[global_idx][1]] = None
                        target_frames[global_idx] = None
                        continue
                    pts = dataset_sf[mf_to_sf_global[global_idx]]['points'].copy()
                    points = np.hstack([pts, np.zeros((pts.shape[0], 1))])
                    
                    az = 60
                    removal_rate = prr[az]
                    attacked_points, state, meta = removal.spoof_frame_ahfr_roadside(
                        points=points,
                        target_box=box,       # aim using the tracked box
                        state=state,          # carries forward temporally
                        p_remove=removal_rate,
                        rng=rng,
                        az_width_deg = az
                    )
                    removal_cache[global_idx] = attacked_points
                    target_frames_anno[global_to_segment[global_idx][1]] = meta


        if(segment != current_segment):
            # temp = target_frames_seg[current_segment]
            # target_frames_anno_seg = {k: (target_frames_anno.get(k) if v is not None else None) for k, v in temp.items()}
            segment_annos = {}
            segment_annos['segment'] = current_segment
            segment_annos['timing_ptt'] = generate_timing_classification(target_frames_anno, 32)
            segment_annos['timing_msf'] = generate_timing_classification(target_frames_anno, 4)
            segment_annos['spoof_annos'] =  target_frames_anno

            dataset_path = f"{save_dir_dataset}/{current_segment}_d.pkl"
            anno_path = f"{save_dir_annos}/{current_segment}_a.pkl"

            with open(dataset_path,"wb") as f:
                pkl.dump(segment_dataset,f)
            with open(anno_path,"wb") as f:
                pkl.dump(segment_annos,f)

            # log.info("Saved: %s", current_segment)

            segment_dataset = [] 
            current_segment = segment

            target_frames = {}
            target_frames_anno = {}
            current_segment = segment
            target_frames = target_frames_global[current_segment].copy()
            target_frames_anno = target_frames_seg[current_segment].copy()
            runs_global = extract_one_runs(target_frames)            
            for run in runs_global:
                subset_g = {k: target_frames[k] for k in run}
                window_min = min(subset_g)
                window_max = max(subset_g)

                tracked_boxes, centers, diag = track_sequence(
                    dataset_mf, start=window_min, end=window_max, desired_range=60.0, target_class=1, negative = True
                )
                ##debug####
                x_center = [center[0] for center in centers]

                mask = ~np.isnan(centers).any(axis=1)

                # 2. Count them
                num_valid_rows = mask.sum()
                print(window_min, window_max, max(x_center), min(x_center), num_valid_rows)

                ##end debug ##

                for key, center, box in zip(subset_g.keys(), centers, tracked_boxes):
                    subset_g[key] = box


                target_frames.update(subset_g)
            removal_cache = {}
            for run in runs_global:
                state = None
                for global_idx in sorted(run):
                    box = target_frames[global_idx]
                    if box is None:
                        target_frames_anno[global_to_segment[global_idx][1]] = None
                        target_frames[global_idx] = None

                        continue
                    pts = dataset_sf[mf_to_sf_global[global_idx]]['points'].copy()
                    points = np.hstack([pts, np.zeros((pts.shape[0], 1))])

                    az = 60
                    removal_rate = prr[az]
                    attacked_points, state, meta = removal.spoof_frame_ahfr_roadside(
                        points=points,
                        target_box=box,       # aim using the tracked box
                        state=state,          # carries forward temporally
                        p_remove=removal_rate,
                        rng=rng,
                        az_width_deg = az
                    )
                    removal_cache[global_idx] = attacked_points
                    target_frames_anno[global_to_segment[global_idx][1]] = meta


        frame_data = {}
        mf_item = dataset_mf[frame].copy()
        frame_data['frame_id'] = mf_item['frame_id']
        frame_data['pose'] = mf_item['poses'][:4]
        frame_data['gt_boxes'] = mf_item['gt_boxes']

        frame_seq = global_to_segment[frame][1]
        four_frame_pts, none_spoofed, single_frame_points = process_4frames(dataset_mf,
                                                                                               dataset_sf, 
                                                                                               frame, 
                                                                                               target_frames, 
                                                                                               frame_seq, 
                                                                                               removal_cache,
                                                                                               mf_to_sf_global)
        
        if(target_frames[frame] is not None and (frame>=40)):
            helpers_ptt.plot_4frame_sequence(four_frame_pts)
            print(frame)
            print(target_frames_anno[global_to_segment[frame][1]]['target_box'])
            input("Press Enter to continue to the next frame...")
#         if(none_spoofed):
#                     # print("clean predictions")
#             load_data_to_gpu(batch)
#             cp_mf_model.eval()
#             with torch.no_grad():
#                 pred, _   = cp_mf_model(batch)
#         else:
#             # print("spoofed predictions")
#             dict_mf_mod = dataset_mf[frame].copy()
#             dict_mf_mod['batch_size']   = 1

#             dict_mf_mod['points'] = four_frame_pts
#                 #! rerun v oxelization and forward pass
#             for k in ['voxels', 'voxel_coords', 'voxel_num_points']:
#                 dict_mf_mod.pop(k, None)
#             dict_mf_mod   = helpers_ptt.inject_gt_names(dict_mf_mod, dataset_mf.class_names)
#             dict_mf_mod   = dataset_mf.prepare_data(dict_mf_mod)

#             load_data_to_gpu(dict_mf_mod)
#             batch_mf_mod = helpers_ptt.convert_to_batch_cp_mf(dict_mf_mod)

#             cp_mf_model.eval()
#             with torch.no_grad():
#                 pred, _   = cp_mf_model(batch_mf_mod)

#         frame_data['pred_boxes'] = pred[0]['pred_boxes']
#         frame_data['pred_scores'] = pred[0]['pred_scores']
#         frame_data['pred_labels'] = pred[0]['pred_labels']
#         frame_data['points'] = single_frame_points
#         frame_data['spoof_gt'] = target_frames[frame]
        
except Exception:
    # log.error("===== EXCEPTION =====")
    # log.error(traceback.format_exc())
    raise  

# if segment_dataset:
#     segment_annos = {}
#     segment_annos['segment'] = current_segment
#     segment_annos['timing_ptt'] = generate_timing_classification(target_frames_anno, 32)
#     segment_annos['timing_msf'] = generate_timing_classification(target_frames_anno, 4)
#     segment_annos['spoof_annos'] =  target_frames_anno

#     dataset_path = f"{save_dir_dataset}/{current_segment}_d.pkl"
#     anno_path = f"{save_dir_annos}/{current_segment}_a.pkl"

#     with open(dataset_path,"wb") as f:
#         pkl.dump(segment_dataset,f)
#     with open(anno_path,"wb") as f:
#         pkl.dump(segment_annos,f)

#     log.info("Saved: %s", current_segment)
    
    # if(target_frames[frame] is not None and (frame == 164 or frame == 191)):
    #     helpers_ptt.plot_4frame_sequence(four_frame_pts)
    #     print(target_frames_anno[global_to_segment[frame][1]]['target_box'])
    #     input("Press Enter to continue to the next frame...")

segment-10203656353524179475_7625_000_7645_000_with_camera_labels
32 63 -54.29630315320037 -77.85406369518387 12
96 127 58.252201301766036 2.438993602814662 32
160 191 -64.16760419627735 -80.69898921151525 12
points shape inside plot function: 112105
points shape inside plot function: 111494
points shape inside plot function: 111749
points shape inside plot function: 110845


40
[-7.14237959e+01 -1.99812988e+01  1.65244031e+00  9.82709715e+00
  3.16470781e+00  3.90000000e+00  2.76064161e+00 -1.90408193e-13
 -3.82748608e-13  1.00000000e+00]


In [10]:
helpers_ptt.plot_4frame_sequence(dataset_mf[400]['points'])

points shape inside plot function: 187671
points shape inside plot function: 187510
points shape inside plot function: 187575
points shape inside plot function: 187742


In [11]:
dataset_mf[400]['gt_boxes']

array([[-3.60870075e+00, -3.60705956e+00,  7.28711486e-01,
         4.53759385e+00,  2.03454206e+00,  1.44000000e+00,
         6.78568264e-02, -2.47197306e-02, -1.42535473e-02,
         1.00000000e+00],
       [-6.33064707e+01, -2.43184203e+01,  4.67006472e+00,
         4.83160407e+00,  1.97385966e+00,  1.51000000e+00,
         4.61727703e-01,  6.16243017e-28, -1.79625280e-27,
         1.00000000e+00],
       [-9.68366631e+00, -5.77910142e+00,  9.91918004e-01,
         5.97567410e-01,  7.35048583e-01,  1.87000000e+00,
         7.82469062e-01,  1.07191257e-01, -8.14038996e-02,
         2.00000000e+00],
       [ 1.56414194e+01, -5.91908665e+00,  1.00014633e+00,
         5.38748329e+00,  2.31595859e+00,  1.87000000e+00,
        -3.01846716e-01, -4.20759088e-02,  1.01888947e-02,
         1.00000000e+00],
       [-2.84142470e+01, -1.14706546e+01,  2.01755972e+00,
         4.19879045e+00,  1.86894655e+00,  1.53000000e+00,
         5.34030461e-01, -1.79688468e-27, -6.17418083e-28,
         1.

In [41]:
target_frames[]